# k-means clustering notebook
Run notebooks in the following order:\
`preprocessing.ipynb -> imputation.R -> dimensionality_reduction.R -> kmeans_clustering.ipynb`

In [ ]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [ ]:
# Imports
from pathlib import Path

import numpy as np
import pandas as pd
from gapstat import gapstat_score
from scipy.stats import chi2_contingency
from sklearn.cluster import KMeans

from utils.bootstrap_utils import BootstrapGridSearch, nan_if_single_cluster
from utils.clustering_utils import predict_clustering_labels
from utils.metrics_utils import (
    adjusted_rand_score,
    association,
    calinski_harabasz_score,
    davies_bouldin_score,
    dunn_index,
    normalized_mutual_info_score,
    silhouette_score,
    variation_of_information,
)
from utils.misc_utils import contingency_matrix
from utils.saving_utils import DatasetBundle, save_object

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Load data
data_discov_full = DatasetBundle.load(Path("data/preprocessed/data_discov_full"))
assert data_discov_full.meta is not None
assert data_discov_full.na is not None
data_valid_full = DatasetBundle.load(Path("data/preprocessed/data_valid_full"))
assert data_valid_full.meta is not None
assert data_valid_full.na is not None

data_discov_mfa = pd.read_parquet(Path("data/dim_red/data_discov_mfa.parquet"))
data_valid_mfa = pd.read_parquet(Path("data/dim_red/data_valid_mfa.parquet"))

# Set outputs folder
output_folder = Path("outputs/kmeans_clustering")
output_folder.mkdir(exist_ok=True)

# Set sample weights
sample_weight_discov = data_discov_full.data["wichtn"].to_numpy()
sample_weight_valid = data_valid_full.data["wichtn"].to_numpy()

In [ ]:
# Create RNG for reproducibility
rng0 = np.random.default_rng(352532)
rng1, rng2, rng3 = rng0.spawn(3)
kmeans_discov = KMeans(random_state=np.random.RandomState(rng0.bit_generator))
kmeans_valid = KMeans(random_state=np.random.RandomState(rng1.bit_generator))

n_boot = 50
param_grid = {"n_clusters": range(2, 51)}
inbag_metrics = {}
oob_metrics = {
    "neg_mean_of_sq": lambda e, X, labels, sample_weight: (
        e.score(X, sample_weight=sample_weight) / (len(X) if sample_weight is None else sample_weight.sum())
    ),
    "dbi": nan_if_single_cluster(
        lambda e, X, labels, sample_weight: davies_bouldin_score(X, labels, sample_weight=sample_weight)
    ),
    "chi": nan_if_single_cluster(
        lambda e, X, labels, sample_weight: calinski_harabasz_score(X, labels, sample_weight=sample_weight)
    ),
    "si": nan_if_single_cluster(
        lambda e, X, labels, sample_weight: silhouette_score(X, labels, sample_weight=sample_weight)
    ),
    "di": nan_if_single_cluster(lambda e, X, labels, sample_weight: dunn_index(X, labels)),
}
agreement_metrics = {
    "ari": lambda x, y, sample_weight: adjusted_rand_score(x, y, sample_weight=sample_weight),
    "nmi": lambda x, y, sample_weight: normalized_mutual_info_score(x, y, sample_weight=sample_weight),
    "vi": lambda x, y, sample_weight: variation_of_information(x, y, sample_weight=sample_weight),
    "assoc": lambda x, y, sample_weight: association(
        contingency_matrix(x, y, sample_weight=sample_weight), correction=False
    ),
    "chi2": lambda x, y, sample_weight: (
        chi2_contingency(contingency_matrix(x, y, sample_weight=sample_weight), correction=False).statistic
    ),
}

# Run bootstrap grid-search loop on discovery data
kmeans_boot_discov = BootstrapGridSearch(
    kmeans_discov,
    param_grid,
    n_boot=n_boot,
    inbag_metrics=inbag_metrics,
    oob_metrics=oob_metrics,
    agreement_metrics=agreement_metrics,
    selection_metric="neg_mean_of_sq",
    tie_break=None,
    use_nonnegative_amendment=True,
    rng=rng2,
)
kmeans_boot_discov = kmeans_boot_discov.fit(data_discov_mfa, sample_weight=sample_weight_discov, verbose=True)

# Fit validation data with only the best parameters from the discovery search
kmeans_boot_valid = BootstrapGridSearch(
    kmeans_valid,
    {k: [v] for k, v in kmeans_boot_discov.best_params_.items()},
    n_boot=n_boot,
    inbag_metrics=inbag_metrics,
    oob_metrics=oob_metrics,
    agreement_metrics=agreement_metrics,
    selection_metric="neg_mean_of_sq",
    tie_break=None,
    use_nonnegative_amendment=True,
    rng=rng3,
)
kmeans_boot_valid = kmeans_boot_valid.fit(data_valid_mfa, sample_weight=sample_weight_valid, verbose=False)

model_discov = kmeans_boot_discov.best_estimator_
model_valid = kmeans_boot_valid.best_estimator_

In [ ]:
# Apply models to predict labels
labels_discov = predict_clustering_labels(model_discov, data_discov_mfa)
labels_valid = predict_clustering_labels(model_valid, data_valid_mfa)

### Compare clusterings
We are dealing with _inferential clustering_, because we are clustering individuals and not variables.\
Method-based validation: The same clustering method is applied to the validation data, and the results are compared between clusterings.\
Result-based validation: The trained cluster algorithm is applied to the validation data, and the results are compared between datasets.\
Cluster stability: The results from the trained and transferred clusterings from the method- and result-based validation steps are compared.

In [ ]:
# Assign metrics directions
metrics_directions = pd.Series(
    {
        "neg_mean_of_sq": "max",
        "dbi": "min",
        "chi": "max",
        "si": "max",
        "di": "max",
        "ari": "max",
        "nmi": "max",
        "vi": "min",
        "assoc": "max",
        "chi2": "max",
    },
    name="direction",
).astype("category")
metrics_grid_search_discov = kmeans_boot_discov.summaries_.merge(
    metrics_directions,
    left_on="metric",
    right_index=True,
    validate="many_to_one",
)
metrics_valid = kmeans_boot_valid.summaries_.merge(
    metrics_directions,
    left_on="metric",
    right_index=True,
    validate="many_to_one",
)

# Compare clusterings
metrics_discov_valid_comparison = (
    metrics_grid_search_discov.merge(pd.DataFrame(kmeans_boot_discov.best_params_, index=pd.Index([0])))
    .set_index(["metric", "scope", "n_clusters", "direction"])
    .join(metrics_valid.set_index(["metric", "scope", "n_clusters", "direction"]), lsuffix="_discov", rsuffix="_valid")
    .reset_index()
    .drop(columns="n_clusters")
)
metrics_discov_valid_comparison

In [ ]:
# Save grid-search results
metrics_grid_search_discov.to_parquet(output_folder / "metrics_grid_search_discov.parquet", index=None)

# Save models
save_object(kmeans_boot_discov, output_folder / "kmeans_boot_discov.pkl")
save_object(model_discov, output_folder / "model_discov.pkl")
save_object(model_valid, output_folder / "model_valid.pkl")

# Save comparison
metrics_discov_valid_comparison.to_parquet(output_folder / "metrics_discov_valid_comparison.parquet", index=False)